# 02 - Feature Extraction

**Project:** The Geometry of Collective Attention  
**Author:** K. Cissé  
**Date:** April 2026  
**Input:** `data/processed/videos_t0.csv`  
**Output:** `data/processed/features.csv`

---

## What this notebook does

Extracts four groups of content-side features for every video:

| Group | Type | Source | Speed |
|---|---|---|---|
| A - Audio | pitch, energy, tempo, ZCR | yt-dlp + librosa | slow (~30s/video) |
| B - Visual | colour, brightness, complexity | thumbnail URL + Pillow | fast (~0.5s/video) |
| C - Text | sentiment, readability, structure | title + description | fast (~0.1s/video) |
| D - Structural | duration, pacing, engagement density | API fields | instant |

---

## Mathematical context

These features will be used in three roles:

1. **Baseline models (M1, M2):** direct predictors of scalar engagement rates  
2. **Sheaf coherence (§4):** inputs to the modality synchronisation maps;  
   we will compute cross-modal correlations (e.g. does audio energy match  
   visual brightness? does speech sentiment match title sentiment?)  
3. **Curvature interpretation:** once we have engagement trajectories in  
   notebook 03, we will ask: which of these features predicts *where* the  
   curvature peaks occur in the video timeline?

The last question is the most novel. A curvature peak at $t^* = 0.6T$  
(60% into the video) is an *event*, and the content features tell us  
what kind of content tends to produce events at that position.

In [1]:
import sys
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm.notebook import tqdm

sys.path.insert(0, str(Path('..').resolve()))
from src.features import (
    extract_all_features,
    extract_text_features,
    extract_visual_features,
    extract_structural_features,
    download_thumbnail,
)

logging.basicConfig(level=logging.WARNING)  # suppress info logs in notebook

DOWNLOAD_AUDIO = True 
AUDIO_DIR      = Path('../data/raw/audio')
THUMB_DIR      = Path('../data/raw/thumbnails')
# ───────────────────────────────────────────────────────────────────────────

AUDIO_DIR.mkdir(parents=True, exist_ok=True)
THUMB_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')
print(f'Audio download: {DOWNLOAD_AUDIO}')

Libraries loaded.
Audio download: True


## 1. Load the collection output

In [2]:
df = pd.read_csv('../data/processed/videos_t0.csv')
print(f'Loaded {len(df)} videos, {len(df.columns)} columns.')
print(f'Categories: {df["our_category_label"].value_counts().to_dict()}')
df.head(2)

Loaded 745 videos, 23 columns.
Categories: {'Science & Education': 80, 'Comedy': 80, 'Meditation & Wellness': 80, 'Finance & Investing': 80, 'Gaming': 80, 'Beauty & Fashion': 80, 'Cooking': 80, 'Personal Vlog': 80, 'Mathematics & Philosophy': 80, 'Political Commentary': 25}


,video_id,title,channel_id,channel_title,our_category,our_category_label,published_at,days_since_publication,view_count,like_count,...,comment_rate,engagement_rate,like_comment_ratio,views_per_day,definition,caption,default_language,thumbnail_url,tags,collected_at
0,KVLTxKyxioA,"The Science of Teaching, Effective Education, ...",UC-RKpEc4eE9PwJaupN91xYQ,Sprouts,science_education,Science & Education,2017-09-15 11:22:21+00:00,3121,814038,19299,...,0.000550,0.024258,43.078125,260.826017,hd,True,en,https://i.ytimg.com/vi/KVLTxKyxioA/maxresdefau...,"['education', 'learning', 'science', 'teachers...",2026-04-02T12:24:24.124032+00:00
1,ldbJU5dwCic,What is the Science of Reading? | Special Educ...,UClU5FPCXMLaMIcfJKo9u3_Q,The Science of Special Education,science_education,Science & Education,2021-03-02 19:00:17+00:00,1856,2258,35,...,0.001329,0.016829,11.666667,1.216595,hd,False,en,https://i.ytimg.com/vi/ldbJU5dwCic/maxresdefau...,"['science of reading', 'teaching reading to st...",2026-04-02T12:24:24.124032+00:00


## 2. Test on one video first

Testing on a single row first, before running the full pipeline.  
This catches import errors, missing libraries, or network issues early.

In [3]:
test_row = df.iloc[0]
print(f'Testing on: "{test_row["title"]}"')
print(f'Category : {test_row["our_category_label"]}')
print()

test_features = extract_all_features(
    row=test_row,
    audio_dir=AUDIO_DIR,
    thumb_dir=THUMB_DIR,
    download_audio_flag=DOWNLOAD_AUDIO,
)

print(f'Extracted {len(test_features)} features.')
print()
for k, v in sorted(test_features.items()):
    if k != 'video_id':
        print(f'  {k:<35} = {v}')

Testing on: "The Science of Teaching, Effective Education, and Great Schools"
Category : Science & Education

Extracted 40 features.

  a_energy_mean                       = 0.10498885065317154
  a_energy_std                        = 0.06575784832239151
  a_music_presence                    = 0.4
  a_pitch_mean                        = 136.03634160107535
  a_pitch_std                         = 98.18854536130041
  a_silence_ratio                     = 0.13641640866873064
  a_spectral_bandwidth                = 2069.703108748107
  a_spectral_centroid                 = 2102.3366525837214
  a_tempo_bpm                         = 123.046875
  a_zcr_mean                          = 0.10302073003337849
  d_duration_minutes                  = 6.35
  d_duration_seconds                  = 381.0
  d_engagement_density                = 51.82939632545932
  d_log_duration                      = 2.582063362911709
  d_log_views                         = 5.910645212111629
  d_pacing_code                 

In [4]:
import pickle
import os
from pathlib import Path

checkpoint_path = Path('../data/processed/checkpoint.pkl')

def load_checkpoint(path):
    """Safely load checkpoint, handling empty/corrupted files."""
    if path.exists():
        size = os.path.getsize(path)
        
        if size == 0:
            print("Checkpoint file is empty. Starting fresh.")
            return []
        
        try:
            with open(path, 'rb') as f:
                data = pickle.load(f)
            
            print(f"✅ Recovered {len(data)} videos from checkpoint")
            return data
        
        except (EOFError, pickle.UnpicklingError):
            print("Checkpoint is corrupted. Starting fresh.")
            return []
    else:
        print("No checkpoint found. Starting from beginning.")
        return []


def save_checkpoint(data, path):
    """Atomic save to prevent corruption."""
    temp_path = path.with_suffix('.tmp')
    
    with open(temp_path, 'wb') as f:
        pickle.dump(data, f)
    
    os.replace(temp_path, path)


# ===== LOAD =====
all_features = load_checkpoint(checkpoint_path)


# ===== FILTER REMAINING DATA =====
if all_features:
    done_ids = {f['video_id'] for f in all_features}
    
    df_remaining = df[~df['video_id'].isin(done_ids)].reset_index(drop=True)
    print(f"Remaining: {len(df_remaining)} videos to process")
else:
    df_remaining = df

✅ Recovered 320 videos from checkpoint
Remaining: 425 videos to process


## 3. Run on all videos

The progress bar shows estimated time remaining.  
If a video fails, `extract_all_features` returns NaN for that group  
rather than crashing; the pipeline is fault-tolerant.

In [5]:
import pickle
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import os

checkpoint_path = Path('../data/processed/checkpoint.pkl')


def save_checkpoint(data, path):
    temp_path = path.with_suffix('.tmp')
    with open(temp_path, 'wb') as f:
        pickle.dump(data, f)
    os.replace(temp_path, path)


# Do not reset if already loaded
# all_features should already come from previous cell

for i, (_, row) in enumerate(tqdm(
    df_remaining.iterrows(), 
    total=len(df_remaining), 
    desc='Extracting features'
)):
    feats = extract_all_features(
        row=row,
        audio_dir=AUDIO_DIR,
        thumb_dir=THUMB_DIR,
        download_audio_flag=DOWNLOAD_AUDIO,
    )
    all_features.append(feats)
    
    # Save every 20 videos
    if (i + 1) % 20 == 0:
        save_checkpoint(all_features, checkpoint_path)
        
        pd.DataFrame(all_features).to_csv(
            '../data/processed/features_partial.csv', index=False
        )
        print(f'💾 Checkpoint saved at {len(all_features)} videos')


# Final save
save_checkpoint(all_features, checkpoint_path)

df_features = pd.DataFrame(all_features)
print(f'\nExtracted {len(df_features.columns)} features for {len(df_features)} videos.')

Extracting features:   5%|▍         | 20/425 [25:19<11:27:29, 101.85s/it]

💾 Checkpoint saved at 340 videos


Extracting features:   8%|▊         | 36/425 [47:21<8:28:18, 78.40s/it]WARNING:src.features:  yt-dlp failed for h1VyskSNq2o: ERROR: Postprocessing: audio conversion failed: Conversion failed!

Extracting features:   9%|▊         | 37/425 [48:39<8:26:09, 78.27s/it]WARNING:src.features:  yt-dlp failed for h8mvUMd73bw: 

ERROR: unable to write data: [Errno 28] No space left on device

Extracting features:   9%|▉         | 38/425 [48:44<6:03:51, 56.41s/it]WARNING:src.features:  Thumbnail download failed for n4xPu72QJsI: [Errno 28] No space left on device

ERROR: unable to write data: [Errno 28] No space left on device

Extracting features:   9%|▉         | 39/425 [48:49<4:22:53, 40.86s/it]WARNING:src.features:  Thumbnail download failed for DcmZpcZl0qg: [Errno 28] No space left on device

Extracting features:   9%|▉         | 39/425 [48:54<8:04:02, 75.24s/it]


OSError: [Errno 28] No space left on device

## 4. Merge with the engagement data

In [ ]:
# Join on video_id
engagement_cols = [
    'video_id', 'our_category', 'our_category_label',
    'view_count', 'like_count', 'comment_count',
    'like_rate', 'comment_rate', 'engagement_rate',
    'days_since_publication', 'views_per_day',
]
engagement_cols = [c for c in engagement_cols if c in df.columns]

df_merged = df[engagement_cols].merge(df_features, on='video_id', how='left')

print(f'Merged dataset: {df_merged.shape[0]} rows × {df_merged.shape[1]} columns')

# Check for missing values by feature group
group_prefixes = {'audio': 'a_', 'visual': 'v_', 'text': 't_', 'structural': 'd_'}
print('\nMissing values by feature group:')
for gname, prefix in group_prefixes.items():
    cols = [c for c in df_merged.columns if c.startswith(prefix)]
    if cols:
        pct_missing = df_merged[cols].isnull().mean().mean() * 100
        print(f'  {gname:<12} ({len(cols)} features): {pct_missing:.1f}% missing')

## 5. Feature visualisations

### 5.1 Text sentiment by category

Title sentiment varies strongly across categories —  
comedy and wellness titles should be positive, political titles negative or polarised.

In [ ]:
plt.rcParams.update({
    'figure.facecolor': '#0d0f14', 'axes.facecolor': '#131620',
    'axes.edgecolor': '#2a3045', 'text.color': '#c8bfa8',
    'axes.labelcolor': '#c8bfa8', 'xtick.color': '#7a8099',
    'ytick.color': '#7a8099', 'grid.color': '#1e2332',
    'grid.linewidth': 0.6, 'font.family': 'monospace', 'font.size': 10,
})

CAT_ORDER = [
    'Science & Education', 'Mathematics & Philosophy', 'Meditation & Wellness',
    'Cooking', 'Beauty & Fashion', 'Personal Vlog',
    'Finance & Investing', 'Gaming', 'Comedy', 'Political Commentary'
]
PALETTE = [
    '#3ecfb8', '#9b7fe8', '#5aa8f0', '#e8a832', '#e05c4a',
    '#78c878', '#c9a84c', '#f0ead8', '#e87850', '#d4635a'
]
cat_colour = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(CAT_ORDER)}

cats_present = df_merged['our_category_label'].unique()

fig, ax = plt.subplots(figsize=(12, 5))

groups = [
    df_merged[df_merged['our_category_label'] == cat]['t_title_sentiment'].dropna()
    for cat in CAT_ORDER if cat in cats_present
]
labels_present = [cat for cat in CAT_ORDER if cat in cats_present]
cols_present   = [cat_colour.get(c, '#7a8099') for c in labels_present]

if groups:
    bp = ax.boxplot(
        groups, patch_artist=True,
        medianprops=dict(color='#f0ead8', linewidth=1.8),
        whiskerprops=dict(color='#7a8099'),
        capprops=dict(color='#7a8099'),
        flierprops=dict(marker='o', markersize=2.5,
                        markerfacecolor='#7a8099', linestyle='none'),
    )
    for patch, col in zip(bp['boxes'], cols_present):
        patch.set_facecolor(col)
        patch.set_alpha(0.45)

    ax.axhline(0, color='#e8a832', linewidth=1, linestyle='--', alpha=0.5,
               label='neutral')
    ax.set_xticks(range(1, len(labels_present) + 1))
    ax.set_xticklabels(
        [l.replace(' & ', '\n& ') for l in labels_present], fontsize=7.5
    )
    ax.set_ylabel('VADER compound sentiment', fontsize=9)
    ax.set_title('Title sentiment by content category', fontsize=11, color='#e8e4dc')
    ax.grid(True, axis='y')
    ax.legend()

plt.tight_layout()
Path('../results/figures').mkdir(parents=True, exist_ok=True)
fig.savefig('../results/figures/02_title_sentiment.png',
            dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()

### 5.2 Visual complexity vs brightness

Each dot is a video, coloured by category.  
We expect beauty/gaming videos to cluster in high-brightness + high-complexity,  
meditation/wellness in low-complexity + moderate brightness.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for cat in CAT_ORDER:
    if cat not in cats_present:
        continue
    sub = df_merged[df_merged['our_category_label'] == cat]
    ax.scatter(
        sub['v_visual_complexity'],
        sub['v_brightness_mean'],
        c=cat_colour.get(cat, '#7a8099'),
        alpha=0.55, s=30, label=cat, edgecolors='none'
    )

ax.set_xlabel('Visual complexity (thumbnail pixel variance)', fontsize=9)
ax.set_ylabel('Brightness (mean luminosity)', fontsize=9)
ax.set_title('Thumbnail visual space — complexity vs brightness', fontsize=11,
             color='#e8e4dc')
ax.legend(fontsize=7, ncol=2, framealpha=0.2)
ax.grid(True)

plt.tight_layout()
fig.savefig('../results/figures/02_visual_scatter.png',
            dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()

### 5.3 Feature correlation heatmap

Which content features are correlated with engagement?  
And which content features are correlated with each other  
(collinearity — a modelling concern)?

In [ ]:
# Select numeric features with low missing rates
target_cols  = ['like_rate', 'comment_rate', 'engagement_rate']
feature_cols = [
    c for c in df_merged.columns
    if c.startswith(('t_', 'v_', 'd_'))
    and df_merged[c].dtype in [float, int, 'float64', 'int64']
    and df_merged[c].isnull().mean() < 0.3
]

if feature_cols:
    corr_cols = target_cols + feature_cols
    corr = df_merged[corr_cols].corr()

    fig, ax = plt.subplots(
        figsize=(max(8, len(corr_cols) * 0.45),
                 max(6, len(corr_cols) * 0.4))
    )

    im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.index)))
    ax.set_xticklabels(corr.columns, rotation=90, fontsize=6)
    ax.set_yticklabels(corr.index, fontsize=6)
    plt.colorbar(im, ax=ax, fraction=0.02)
    ax.set_title('Feature correlation matrix', fontsize=11, color='#e8e4dc')

    # Highlight target rows
    for i, col in enumerate(corr.columns):
        if col in target_cols:
            ax.add_patch(plt.Rectangle((i - 0.5, -0.5), 1, len(corr), 
                                        fill=False, edgecolor='#e8a832',
                                        lw=1.5))

    plt.tight_layout()
    fig.savefig('../results/figures/02_feature_correlation.png',
                dpi=150, bbox_inches='tight', facecolor='#0d0f14')
    plt.show()
else:
    print('No numeric features available yet. Run with audio for full matrix.')

### 5.4 Top features correlated with like_rate

Horizontal bar chart of absolute correlation with `like_rate`.  
This gives the first answer to: **which content features matter most?**

In [ ]:
if feature_cols:
    corr_with_like = (
        df_merged[feature_cols + ['like_rate']]
        .corr()['like_rate']
        .drop('like_rate')
        .abs()
        .sort_values(ascending=False)
        .head(20)
    )

    fig, ax = plt.subplots(figsize=(8, 6))
    bars = ax.barh(
        range(len(corr_with_like)),
        corr_with_like.values,
        color='#3ecfb8', alpha=0.7, edgecolor='none'
    )
    ax.set_yticks(range(len(corr_with_like)))
    ax.set_yticklabels(corr_with_like.index, fontsize=8)
    ax.set_xlabel('|Pearson correlation| with like_rate', fontsize=9)
    ax.set_title('Top 20 features by correlation with like_rate',
                 fontsize=11, color='#e8e4dc')
    ax.grid(True, axis='x')
    ax.invert_yaxis()

    plt.tight_layout()
    fig.savefig('../results/figures/02_top_features.png',
                dpi=150, bbox_inches='tight', facecolor='#0d0f14')
    plt.show()

    print('Top 5 features by correlation with like_rate:')
    print(corr_with_like.head(5).to_string())
else:
    print('No features to rank yet.')

## 6. Save the feature table

In [ ]:
out_path = Path('../data/processed/features.csv')
df_merged.to_csv(out_path, index=False)

print(f'Saved {df_merged.shape[0]} rows x {df_merged.shape[1]} columns -> {out_path}')

# Summary for the record
n_audio    = len([c for c in df_merged.columns if c.startswith('a_')])
n_visual   = len([c for c in df_merged.columns if c.startswith('v_')])
n_text     = len([c for c in df_merged.columns if c.startswith('t_')])
n_struct   = len([c for c in df_merged.columns if c.startswith('d_')])

print(f'\nFeature counts:')
print(f'  Audio      : {n_audio}')
print(f'  Visual     : {n_visual}')
print(f'  Text       : {n_text}')
print(f'  Structural : {n_struct}')
print(f'  Total      : {n_audio + n_visual + n_text + n_struct}')

In [ ]:
import pandas as pd
from pathlib import Path
from src.features import download_audio, extract_audio_features

df = pd.read_csv('../data/processed/features.csv')

# Find videos where audio extraction failed
failed = df[df['a_pitch_mean'].isna()]['video_id'].tolist()
print(f'{len(failed)} videos need audio retry')

audio_dir = Path('../data/raw/audio')
results = []

for vid_id in failed:
    print(f'Retrying: {vid_id}')
    path = download_audio(vid_id, audio_dir)
    if path:
        feats = extract_audio_features(path)
        feats['video_id'] = vid_id
        results.append(feats)

# Update the dataframe with successful retries
if results:
    retry_df = pd.DataFrame(results)
    for _, row in retry_df.iterrows():
        mask = df['video_id'] == row['video_id']
        for col in row.index:
            if col != 'video_id' and f'a_{col}' in df.columns:
                df.loc[mask, f'a_{col}'] = row[col]
    df.to_csv('../data/processed/features.csv', index=False)
    print(f'Updated {len(results)} videos with recovered audio features')

## 7. What is still missing, and why it matters

This notebook produces the content-side features.  
What is intentionally absent:

**The engagement trajectory $\mathbf{e}(t)$** is not yet here because it  
requires re-querying the same videos at $t_1, t_2, t_3$ — done in notebook 03.  

**Audio features** require `DOWNLOAD_AUDIO = True` — run this as a  
separate long job, ideally overnight:

```bash
jupyter nbconvert --to notebook --execute notebooks/02_feature_extraction.ipynb \
    --ExecutePreprocessor.timeout=7200
```

**The key question this notebook raises:**  
Look at the top-5 features correlated with `like_rate`.  
If a *structural* feature (e.g. `d_log_duration`) ranks above any  
*content* feature, it means the baseline model (M1) is already capturing  
most of the variance, and the richer features add less than expected.  
This would motivate a stronger focus on the trajectory geometry as the  
genuinely new explanatory layer — exactly the motivation for models M3 and M4.